# Глава 2. Создание и наполнение базы данных

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Формируем структуру подключения к БД
connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="*UHB5rdx",
)
# Создаем движок подключения Engine и передаем в него готовый объект URL
engine = create_engine(connection_url)

# Активируем магические команды JupySQL в текущем ноутбуке
%load_ext sql

# Задать лимит отображения количества строк ( default=10, None осторожно)
%config SqlMagic.displaylimit = 100
# Отключить вывод строки подключения 'Running query in 'mysql+pymysql...'
%config SqlMagic.displaycon = False
# Отключить вывод количества затронутых строк `10 rows affected.`
# %config SqlMagic.feedback = False

# Передаем наш движок в JupySQL
%sql engine

# Вывод подтверждения
print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


:::{hint} Типы данных
:class: dropdown
:open: false
:icon: false

```markdown
# Символьный (текстовый, строковый)

| Тип        | Описание                | Макс байт     |
| ---------- | ----------------------- | ------------- |
| CHAR       | Постоянная длина        | 255           |
| VARCHAR    | Переменная длина        | 65 535        |
| mediumtext |                         | 16 777 215    |
| longtext   |                         | 4 294 967 295 |
| ENUM       | Проверочное ограничение |               |
```

```markdown
# Целочисленный

| Тип       | Знаковый диапазон                  | UNSIGNED              |
| --------- | ---------------------------------- | --------------------- |
| INT       | от -2 147 483 648 до 2 147 483 647 | от 0 до 4 294 967 295 |
| BIGINT    | от -2'63 до 2'63-1                 | от 0 до 2'64-1        |
| mediumint | от -8 388 608 до 8 388 607         | от 0 до 16 777 215    |
| smallint  | от -32 768 до 32 767               | от 0 до 65 535        |
| tinyint   | от -128 до 127                     | от 0 до 255           |
```

```markdown
# C плавающей точкой

| Тип          |                           |
| ------------ | ------------------------- |
| FLOAT(p, s)  | ~ 7 знаков после запятой  |
| double(p, s) | ~ 15 знаков после запятой |

- `p` (precision) – точность: общее количество допустимых цифр слева и справа от десятичной точки;
- `s` (scale) – масштаб: количество допустимых цифр справа от десятичной точки
```

```markdown
# Временной

| Тип       | Формат              |
| --------- | ------------------- |
| date      | YYYY-MM-DD          |
| datetime  | YYYY-MM-DD HH:MI:SS |
| timestamp | YYYY-MM-DD HH:MI:SS |
| year      | YYYY                |
| time      | HHH:MI:SS           |
```

:::

:::{note} Таблица person
:class: dropdown
:open: false
:icon: false

```text
| Столбец     | Тип                | Допустимые значения |
| ----------- | ------------------ | ------------------- |
| person_id   | int(unsigned)      |                     |
| first_name  | varchar(20)        |                     |
| last_name   | varchar(20)        |                     |
| eye_color   | char(2)            | BL, BR, GR          |
| birth_date  | date               |                     |
| street      | varchar(30)        |                     |
| city        | varchar(20)        |                     |
| state       | varchar(20)        |                     |
| country     | varchar(20)        |                     |
| postal_code | varchar(20)        |                     |
```
Заменил `smalllint` –> `int`
:::

markdown
```sql
CREATE TABLE person(
    person_id INT UNSIGNED,
    fname VARCHAR(20),
    lname VARCHAR(20),
    eye_color ENUM('BR', 'BL', 'GR'),
    birth_date DATE,
    street VARCHAR(30),
    city VARCHAR(20),
    state VARCHAR(20),
    country VARCHAR(20),
    postal_code VARCHAR(20),
    CONSTRAINT pk_person PRIMARY KEY (person_id)
);
```

In [2]:
%%sql
CREATE TABLE person(
    person_id INT UNSIGNED,
    fname VARCHAR(20),
    lname VARCHAR(20),
    eye_color ENUM('BR', 'BL', 'GR'),
    birth_date DATE,
    street VARCHAR(30),
    city VARCHAR(20),
    state VARCHAR(20),
    country VARCHAR(20),
    postal_code VARCHAR(20),
    CONSTRAINT pk_person PRIMARY KEY (person_id)
);

++
||
++
++

:::{hint} Best practice
:class: dropdown
:open: false

При создании таблицы _best practice_ сразу включать функцию авто-инкремента для столбца первичного ключа `INT PRIMARY KEY AUTO_INCREMENT`
```sql
CREATE TABLE person(
    person_id INT PRIMARY KEY AUTO_INCREMENT,
    fname VARCHAR(20),
    lname VARCHAR(20),
    -- ...
    postal_code VARCHAR(20)
);
:::

`DESCRIBE` – посмотреть определение созданной таблицы
```sql
DESC person;
```

In [3]:
%%sql
DESC person;

10 rows affected.

Field,Type,Null,Key,Default,Extra
person_id,int unsigned,NO,PRI,None,
fname,varchar(20),YES,,None,
lname,varchar(20),YES,,None,
eye_color,"enum('BR','BL','GR')",YES,,None,
birth_date,date,YES,,None,
street,varchar(30),YES,,None,
city,varchar(20),YES,,None,
state,varchar(20),YES,,None,
country,varchar(20),YES,,None,
postal_code,varchar(20),YES,,None,


markdown
```sql
CREATE TABLE favorite_food(
    person_id INT UNSIGNED,
    food VARCHAR(20),
    CONSTRAINT pk_favorite_food PRIMARY KEY (person_id, food),
    CONSTRAINT fk_favorite_food FOREIGN KEY (person_id) REFERENCES person (person_id)
);
```

In [4]:
%%sql
CREATE TABLE favorite_food(
    person_id INT UNSIGNED,
    food VARCHAR(20),
    CONSTRAINT pk_favorite_food PRIMARY KEY (person_id, food),
    CONSTRAINT fk_favorite_food FOREIGN KEY (person_id) REFERENCES person (person_id)
);

++
||
++
++

markdown
```sql
DESC favorite_food;
```

In [5]:
%%sql
DESC favorite_food;

2 rows affected.

Field,Type,Null,Key,Default,Extra
person_id,int unsigned,NO,PRI,None,
food,varchar(20),NO,PRI,None,


## Заполнение и изменение таблиц

### Генерация данных числовых ключей

In [8]:
%%sql
ALTER TABLE person MODIFY person_id INT UNSIGNED AUTO_INCREMENT;

RuntimeError: (pymysql.err.OperationalError) (1833, "Cannot change column 'person_id': used in a foreign key constraint 'fk_favorite_food' of table 'sakila.favorite_food'")
[SQL: ALTER TABLE person MODIFY person_id INT UNSIGNED AUTO_INCREMENT;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


::::{error} Следуя за автором получаем ошибку
:class: dropdown
:open: true

СУБД (MySQL) не позволяет изменять свойства колонки, если на неё уже ссылается внешний ключ (FK) из другой таблицы. Когда пытаемся добавить `AUTO_INCREMENT` к `person_id` в таблице `person`, база данных блокирует это действие, так как таблица `favorite_food` уже жестко связана с исходным определением этого поля.

Самый простой и быстрый способ в MySQL – временно отключить проверку внешних ключей на время выполнения команды.

```sql
-- 1. Временно отключаем проверку внешних ключей
SET FOREIGN_KEY_CHECKS = 0;

-- 2. Выполняем изменение (добавляем AUTO_INCREMENT)
ALTER TABLE person MODIFY person_id INT UNSIGNED AUTO_INCREMENT;

-- 3. Включаем проверку внешних ключей обратно
SET FOREIGN_KEY_CHECKS = 1;
```

:::{hint} Best practice
:class: dropdown
:open: false

При создании таблицы сразу указывать `AUTO_INCREMENT` для `PK`:
```sql
person_id INT PRIMARY KEY AUTO_INCREMENT
```
:::

::::

In [2]:
%%sql
SET FOREIGN_KEY_CHECKS = 0;

++
||
++
++

In [4]:
%%sql
ALTER TABLE person MODIFY person_id INT UNSIGNED AUTO_INCREMENT;

++
||
++
++

In [5]:
%%sql
SET FOREIGN_KEY_CHECKS = 1;

++
||
++
++

In [6]:
%%sql
DESC person;

10 rows affected.

Field,Type,Null,Key,Default,Extra
person_id,int unsigned,NO,PRI,None,auto_increment
fname,varchar(20),YES,,None,
lname,varchar(20),YES,,None,
eye_color,"enum('BR','BL','GR')",YES,,None,
birth_date,date,YES,,None,
street,varchar(30),YES,,None,
city,varchar(20),YES,,None,
state,varchar(20),YES,,None,
country,varchar(20),YES,,None,
postal_code,varchar(20),YES,,None,


### Добавление данных

In [7]:
%%sql
INSERT INTO person
    (person_id, fname, lname, eye_color, birth_date)
VALUES (null, 'William', 'Turner', 'BR', '1972-05-27');

1 rows affected.

++
||
++
++

In [11]:
%%sql
SELECT person_id, fname, lname, eye_color, birth_date
FROM person;

1 rows affected.

person_id,fname,lname,eye_color,birth_date
1,William,Turner,BR,1972-05-27


In [12]:
%%sql
SELECT person_id, fname, lname, eye_color, birth_date
FROM person
WHERE person_id = 1;

1 rows affected.

person_id,fname,lname,eye_color,birth_date
1,William,Turner,BR,1972-05-27


In [13]:
%%sql
SELECT person_id, fname, lname, eye_color, birth_date
FROM person
WHERE lname = 'Turner';

1 rows affected.

person_id,fname,lname,eye_color,birth_date
1,William,Turner,BR,1972-05-27


In [14]:
%%sql
INSERT INTO favorite_food (person_id, food)
VALUES (1, 'pizza');

1 rows affected.

++
||
++
++

In [15]:
%%sql
INSERT INTO favorite_food (person_id, food)
VALUES (1, 'cookies');

1 rows affected.

++
||
++
++

In [16]:
%%sql
INSERT INTO favorite_food (person_id, food)
VALUES (1, 'nachos');

1 rows affected.

++
||
++
++

In [17]:
%%sql
SELECT food
FROM favorite_food
WHERE person_id = 1
ORDER BY food;

3 rows affected.

food
cookies
nachos
pizza


In [18]:
%%sql
INSERT INTO person
    (person_id, fname, lname, eye_color, birth_date,
    street, city, state, country, postal_code)
VALUES
    (null, 'Susan', 'Smith', 'BL', '1975-11-02',
    '23 Maple St.', 'Arlington', 'VA', 'USA', '20220');

1 rows affected.

++
||
++
++

In [21]:
%%sql
SELECT person_id, fname, lname, birth_date
FROM person;

2 rows affected.

person_id,fname,lname,birth_date
1,William,Turner,1972-05-27
2,Susan,Smith,1975-11-02


::::{tip} Генерация в XML
:class: dropdown
:open: false

Приведенный в книге код принадлежит и выполняется в Microsoft SQL Server:
```sql
SELECT * FROM favorite_food
FOR XML AUTO, ELEMENTS
```

Мы работаем в MySQL, где синтаксис для работы с XML совершенно другой. К тому же, в современных версиях MySQL для работы со структурами данных гораздо чаще используется формат JSON, т.к. встроенная поддержка XML в MySQL довольно ограничена.

**Выгрузка в JSON** (рекомендуемая для MySQL)
```sql
%%sql
SELECT JSON_OBJECT('person_id', person_id, 'food', food) AS json_result 
FROM favorite_food;
```

:::{hint} Что дальше? 
:class: dropdown
:open: false

После выполнения запроса данные существуют только в оперативной памяти. Можем перехватить их и передать напрямую в Python, чтобы сохранить их в файл (JSON, XML, CSV) или использовать для анализа.

```python
# 1. Превращаем результат SQL-запроса в DataFrame
df = _.DataFrame()

# 2. Извлекаем только колонку с JSON и сохраняем в файл
with open('favorite_food_output.json', 'w', encoding='utf-8') as f:
    # Соединяем все строки в один красивый JSON-массив
    json_string = "[" + ",".join(df['json_result']) + "]"
    f.write(json_string)

print("Файл favorite_food_output.json успешно сохранен!")
```

После выполнения файл _favorite_food_output.json_ появится в левой панели Jupyter Lab (File Browser) в _текущей_ папке.

Или в _дочерней_, если сразу указать относительный путь к _**заранее**_ созданной папке:

```python
with open('data/favorite_food_output.json', 'w', encoding='utf-8') as f:
```
:::

::::

### Изменение данных

In [26]:
%%sql
UPDATE person
SET street = '1225 Tremont St.',
    city = 'Boston',
    state = 'MA',
    country = 'USA',
    postal_code = '02138'
WHERE person_id = 1;

1 rows affected.

++
||
++
++

In [27]:
%%sql
SELECT * FROM person;

2 rows affected.

person_id,fname,lname,eye_color,birth_date,street,city,state,country,postal_code
1,William,Turner,BR,1972-05-27,1225 Tremont St.,Boston,MA,USA,02138
2,Susan,Smith,BL,1975-11-02,23 Maple St.,Arlington,VA,USA,20220


### Удаление данных

In [28]:
%%sql
DELETE FROM person
WHERE person_id = 2;

1 rows affected.

++
||
++
++

In [29]:
%%sql
SELECT * FROM person;

1 rows affected.

person_id,fname,lname,eye_color,birth_date,street,city,state,country,postal_code
1,William,Turner,BR,1972-05-27,1225 Tremont St.,Boston,MA,USA,02138


## Когда хорошие инструкции становятся плохими

### Не уникальный первичный ключ

In [30]:
%%sql
INSERT INTO person
    (person_id, fname, lname, eye_color, birth_date)
VALUES (1, 'Charles', 'Fulton', 'GR', '1968-01-15');

RuntimeError: (pymysql.err.IntegrityError) (1062, "Duplicate entry '1' for key 'person.PRIMARY'")
[SQL: INSERT INTO person
    (person_id, fname, lname, eye_color, birth_date)
VALUES (1, 'Charles', 'Fulton', 'GR', '1968-01-15');]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


### Несуществующий внешний ключ

In [31]:
%%sql
INSERT INTO favorite_food (person_id, food)
VALUES (999, 'lasagna');

RuntimeError: (pymysql.err.IntegrityError) (1452, 'Cannot add or update a child row: a foreign key constraint fails (`sakila`.`favorite_food`, CONSTRAINT `fk_favorite_food` FOREIGN KEY (`person_id`) REFERENCES `person` (`person_id`))')
[SQL: INSERT INTO favorite_food (person_id, food)
VALUES (999, 'lasagna');]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


### Нарушения значения столбцов

In [34]:
%%sql
UPDATE person
SET eye_color = 'ZZ'
WHERE person_id = 1;

RuntimeError: (pymysql.err.DataError) (1265, "Data truncated for column 'eye_color' at row 1")
[SQL: UPDATE person
SET eye_color = 'ZZ'
WHERE person_id = 1;]
(Background on this error at: https://sqlalche.me/e/20/9h9h)


### Некорректное преобразование данных

In [35]:
%%sql
UPDATE person
SET birth_date = 'DEC-21-1980'
WHERE person_id = 1;

RuntimeError: (pymysql.err.OperationalError) (1292, "Incorrect date value: 'DEC-21-1980' for column 'birth_date' at row 1")
[SQL: UPDATE person
SET birth_date = 'DEC-21-1980'
WHERE person_id = 1;]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [36]:
%%sql
UPDATE person
SET birth_date = str_to_date('DEC-21-1980', '%b-%d-%Y')
WHERE person_id = 1;

1 rows affected.

++
||
++
++

In [37]:
%%sql
SELECT person_id, fname, lname, birth_date
FROM person;

1 rows affected.

person_id,fname,lname,birth_date
1,William,Turner,1980-12-21


:::{seealso} CheatSheet для преобразования строк в дату и время
:class: dropdown
:open: false

```markdown
%а Краткое имя дня недели - Sun, Mon, ...
%b Краткое имя месяца — Jan, Feb, ...
%с Числовое значение месяца (0..11)
%d Числовое значение дня месяца (00..31)
%f Число микросекунд (000000..999999)
%Н Час дня в 24-часовом формате (00..23)
%h Час дня в 12-часовом формате (01..12)
%i Минуты в часе (00..59)
%j День года (001..366)
%М Полное имя месяца (January..December)
%m Числовое значение месяца
%р AM или РМ
%s Число секунд (00..59)
%W Полное имя дня недели (Sunday..Saturday)
%w Числовое значение дня недели (0=Sunday..6=Saturday)
%Y Значение года (четыре цифры)
```

:::

## База данных Sakila

In [39]:
%%sql
SHOW TABLES;

25 rows affected.

Tables_in_sakila
actor
actor_info
address
category
city
country
customer
customer_list
favorite_food
film


In [40]:
%%sql
DROP TABLE favorite_food;

++
||
++
++

In [41]:
%%sql
DROP TABLE person;

++
||
++
++

In [42]:
%%sql
SHOW TABLES;

23 rows affected.

Tables_in_sakila
actor
actor_info
address
category
city
country
customer
customer_list
film
film_actor


In [43]:
%%sql
DESC customer;

9 rows affected.

Field,Type,Null,Key,Default,Extra
customer_id,smallint unsigned,NO,PRI,None,auto_increment
store_id,tinyint unsigned,NO,MUL,None,
first_name,varchar(45),NO,,None,
last_name,varchar(45),NO,MUL,None,
email,varchar(50),YES,,None,
address_id,smallint unsigned,NO,MUL,None,
active,tinyint(1),NO,,1,
create_date,datetime,NO,,None,
last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP
